# Gráficos do fitointer-sus

Figuras da análise de risco de interação entre plantas medicinais do SUS e
fármacos da RENAME.

As figuras saem em `resultados/7_graficos/`, em PNG (300 dpi) para o relatório e
em SVG para edição. Os dados vêm de `resultados/6_analise/`, geradas por
`scripts/6_analise/analisar.py`.

Rode as células em ordem.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

# Acha a raiz do projeto, rodando o notebook de qualquer pasta
BASE = Path.cwd()
while not (BASE / "resultados").is_dir() and BASE != BASE.parent:
    BASE = BASE.parent

ANALISE = BASE / "resultados" / "6_analise"
INTERACOES = BASE / "resultados" / "4_interacoes"
SAIDA = BASE / "resultados" / "7_graficos"
SAIDA.mkdir(parents=True, exist_ok=True)

print("Projeto em:", BASE)

## Cores e estilo

A paleta segue uma escala ordinal de azul para o nível de evidência (claro a
escuro = baixa a alta) e cores categóricas para a origem da evidência. Os
valores foram validados para contraste e para daltonismo.

In [ ]:
SUPERFICIE = "#fcfcfb"
TINTA = "#0b0b0b"
TINTA_2 = "#52514e"
MUDO = "#898781"
GRADE = "#e1e0d9"
EIXO = "#c3c2b7"

# Escala ordinal: quanto mais escuro, mais forte a evidência
NIVEL = {"baixa": "#86b6ef", "media": "#2a78d6", "alta": "#104281"}
NIVEL_ROTULO = {"baixa": "Baixa", "media": "Média", "alta": "Alta"}

# Cores categóricas: identidade, não ordem
ORIGEM = {
    "triagem_pubchem": "#2a78d6",
    "ensaio_dedicado": "#eb6834",
    "literatura": "#1baf7a",
}
ORIGEM_ROTULO = {
    "triagem_pubchem": "Triagem automatizada (PubChem)",
    "ensaio_dedicado": "Ensaio dedicado",
    "literatura": "Literatura",
}

plt.rcParams.update({
    "figure.facecolor": SUPERFICIE,
    "axes.facecolor": SUPERFICIE,
    "savefig.facecolor": SUPERFICIE,
    "font.family": "sans-serif",
    "font.size": 10,
    "text.color": TINTA,
    "axes.labelcolor": TINTA_2,
    "axes.edgecolor": EIXO,
    "xtick.color": MUDO,
    "ytick.color": MUDO,
    "xtick.labelcolor": TINTA_2,
    "ytick.labelcolor": TINTA_2,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def tinta_rotulo(nivel):
    """Texto escuro sobre o azul claro, claro sobre os escuros."""
    return TINTA if nivel == "baixa" else "#ffffff"


def salvar(fig, nome):
    """Salva em PNG (relatório) e SVG (edição)."""
    for ext in ("png", "svg"):
        fig.savefig(SAIDA / f"{nome}.{ext}", dpi=300, bbox_inches="tight")
    print("salvo:", nome)

## Figura 1 — Plantas e fármacos que interagem

Cada célula é um par planta × fármaco. A cor indica a força da evidência. É a
figura principal do trabalho.

In [ ]:
matriz = pd.read_csv(INTERACOES / "matriz_planta_farmaco.csv", encoding="utf-8-sig")
matriz = matriz.set_index("planta")

ordem = {"baixa": 1, "media": 2, "alta": 3}
valores = matriz.apply(lambda coluna: coluna.map(ordem))

# Mais evidência primeiro, nas duas direções
valores = valores.loc[valores.sum(axis=1).sort_values(ascending=False).index]
valores = valores[valores.sum().sort_values(ascending=False).index]

mapa = ListedColormap([NIVEL["baixa"], NIVEL["media"], NIVEL["alta"]])
mapa.set_bad(SUPERFICIE)
limites = BoundaryNorm([0.5, 1.5, 2.5, 3.5], mapa.N)

fig, ax = plt.subplots(figsize=(11, 5))
ax.pcolormesh(np.ma.masked_invalid(valores.values), cmap=mapa, norm=limites,
              edgecolors=SUPERFICIE, linewidth=2)

ax.set_xticks(np.arange(len(valores.columns)) + 0.5)
ax.set_xticklabels(valores.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(np.arange(len(valores.index)) + 0.5)
ax.set_yticklabels(valores.index, fontsize=9)
ax.invert_yaxis()
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)

ax.set_title("Pares planta × fármaco, por força da evidência",
             loc="left", fontsize=13, pad=14, color=TINTA)
ax.legend(handles=[Patch(facecolor=NIVEL[n], label=NIVEL_ROTULO[n])
                   for n in ("alta", "media", "baixa")],
          loc="upper left", bbox_to_anchor=(0, -0.3), ncol=3,
          frameon=False, fontsize=9, handlelength=1.2, handleheight=1.2)

salvar(fig, "fig1_mapa_planta_farmaco")
plt.show()

## Figura 2 — Fármacos afetados por planta

Quantos fármacos da RENAME cada planta pode afetar, separados pela força da
evidência. Só os ativos principais de cada planta entram na contagem.

In [ ]:
plantas = pd.read_csv(ANALISE / "ranking_plantas.csv", encoding="utf-8-sig")
plantas = plantas.set_index("planta")[["pares_alta", "pares_media", "pares_baixa"]]
# Ordena pela força da evidência, não pelo total: quem tem mais pares de nível
# alto vem primeiro (barh desenha de baixo para cima)
plantas = plantas.sort_values(["pares_alta", "pares_media", "pares_baixa"])

fig, ax = plt.subplots(figsize=(9, 5))
inicio = np.zeros(len(plantas))
for chave, coluna in zip(("alta", "media", "baixa"),
                         ("pares_alta", "pares_media", "pares_baixa")):
    larguras = plantas[coluna].values
    ax.barh(plantas.index, larguras, left=inicio, height=0.62,
            color=NIVEL[chave], label=NIVEL_ROTULO[chave],
            edgecolor=SUPERFICIE, linewidth=2)
    # Rótulo direto dentro do segmento, quando couber
    for y, (x, largura) in enumerate(zip(inicio, larguras)):
        if largura >= 3:
            ax.text(x + largura / 2, y, str(int(largura)), ha="center", va="center",
                    color=tinta_rotulo(chave), fontsize=9)
    inicio += larguras

ax.set_xlabel("fármacos da RENAME")
ax.tick_params(length=0)
ax.spines["left"].set_color(EIXO)
ax.spines["bottom"].set_visible(False)
ax.set_xticks([])
ax.set_title("Fármacos afetados por planta", loc="left", fontsize=13, pad=14, color=TINTA)
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.06), ncol=3, frameon=False,
          fontsize=9, handlelength=1.2, handleheight=1.2)

salvar(fig, "fig2_farmacos_por_planta")
plt.show()

## Figura 3 — Enzimas envolvidas

Por onde passam as interações. A CYP3A4 concentra quase tudo, porque é a enzima
que metaboliza a maioria dos fármacos e a que tem mais substratos sensíveis na
tabela da FDA.

In [ ]:
enzimas = pd.read_csv(ANALISE / "resumo_enzimas.csv", encoding="utf-8-sig")
enzimas = enzimas.set_index("enzima")[["pares_alta", "pares_media", "pares_baixa"]]
enzimas = enzimas.loc[enzimas.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(8, 2.2))
inicio = np.zeros(len(enzimas))
for chave, coluna in zip(("alta", "media", "baixa"),
                         ("pares_alta", "pares_media", "pares_baixa")):
    larguras = enzimas[coluna].values
    ax.barh(enzimas.index, larguras, left=inicio, height=0.55,
            color=NIVEL[chave], label=NIVEL_ROTULO[chave],
            edgecolor=SUPERFICIE, linewidth=2)
    for y, (x, largura) in enumerate(zip(inicio, larguras)):
        if largura >= 6:
            ax.text(x + largura / 2, y, str(int(largura)), ha="center", va="center",
                    color=tinta_rotulo(chave), fontsize=9)
    inicio += larguras

for y, total in enumerate(enzimas.sum(axis=1).values):
    ax.text(total + 2, y, str(int(total)), va="center", color=TINTA_2, fontsize=9)

ax.set_xlabel("pares planta × fármaco")
ax.set_xticks([])
ax.tick_params(length=0)
ax.spines["left"].set_color(EIXO)
ax.spines["bottom"].set_visible(False)
ax.set_title("Pares por enzima", loc="left", fontsize=13, pad=14, color=TINTA)
ax.legend(loc="lower right", frameon=False, fontsize=9,
          handlelength=1.2, handleheight=1.2)

salvar(fig, "fig3_pares_por_enzima")
plt.show()

## Figura 4 — De onde vem a evidência

Registros de atividade considerados ativos, por origem. A triagem automatizada
do PubChem cobre muitos compostos, mas é menos confiável que um ensaio feito
para medir aquela enzima.

In [ ]:
origem = pd.read_csv(ANALISE / "origem_evidencia_total.csv", encoding="utf-8-sig")
origem = origem.set_index("origem")[["registros", "compostos"]]
origem = origem.loc[origem["registros"].sort_values().index]

fig, ax = plt.subplots(figsize=(8, 2.6))
barras = ax.barh([ORIGEM_ROTULO[i] for i in origem.index], origem["registros"],
                 height=0.55, color=[ORIGEM[i] for i in origem.index])

for barra, (registros, compostos) in zip(barras, origem.values):
    ax.text(barra.get_width() + 3, barra.get_y() + barra.get_height() / 2,
            f"{registros} registros · {compostos} compostos",
            va="center", color=TINTA_2, fontsize=9)

ax.set_xlim(0, origem["registros"].max() * 1.45)
ax.set_xticks([])
ax.tick_params(length=0)
ax.spines["left"].set_color(EIXO)
ax.spines["bottom"].set_visible(False)
ax.set_title("Origem da evidência de atividade", loc="left", fontsize=13, pad=14, color=TINTA)

salvar(fig, "fig4_origem_evidencia")
plt.show()

## Figura 5 — Onde faltam dados

Proporção dos ativos principais de cada planta que não tem nenhuma atividade
medida nas enzimas do estudo. É o mapa do que ainda precisa ser buscado na
literatura.

In [ ]:
lacunas = pd.read_csv(ANALISE / "lacunas.csv", encoding="utf-8-sig")
lacunas = lacunas.sort_values("sem_dados_pct")

fig, ax = plt.subplots(figsize=(8, 4.6))
barras = ax.barh(lacunas["planta"], lacunas["sem_dados_pct"], height=0.62,
                 color=NIVEL["media"])

for barra, (sem, total) in zip(barras, lacunas[["sem_dados", "ativos_principais"]].values):
    largura = barra.get_width()
    ax.text(largura + 1.5, barra.get_y() + barra.get_height() / 2,
            f"{int(sem)} de {int(total)}", va="center", color=TINTA_2, fontsize=9)

ax.set_xlim(0, 112)
ax.set_xlabel("% dos ativos principais sem atividade medida")
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0", "25", "50", "75", "100%"])
ax.tick_params(length=0)
ax.spines["left"].set_color(EIXO)
ax.spines["bottom"].set_visible(False)
ax.set_title("Ativos principais sem dado de atividade", loc="left",
             fontsize=13, pad=14, color=TINTA)

salvar(fig, "fig5_lacunas")
plt.show()

## Números do estudo

Para conferir junto com as figuras e para o texto do relatório.

In [ ]:
numeros = pd.read_csv(ANALISE / "numeros_do_estudo.csv", encoding="utf-8-sig")
numeros